# 03 Run Zero-Shot Mutant Design

This notebook is a lightweight orchestrator around `run_zero_shot_mutant_design_workflow`, with heavy logic moved into `src/agentic_protein_design/workflows/run_zero_shot_mutant_design/`.


In [1]:
from pathlib import Path
import sys

# Section 1: Resolve repo root from either project root or notebooks/ cwd.
cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd

# Section 2: Add repo and src roots to sys.path for imports.
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
src_root = repo_root / "src"
if src_root.exists() and str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

print("repo_root:", repo_root)
print("src_root:", src_root)


repo_root: /Users/charmainechia/Documents/projects/agentic-protein-design
src_root: /Users/charmainechia/Documents/projects/agentic-protein-design/src


In [2]:
from pprint import pprint

from agentic_protein_design.workflows.run_zero_shot_mutant_design.config import build_user_inputs
from agentic_protein_design.workflows.run_zero_shot_mutant_design.workflow import run_zero_shot_mutant_design_workflow


In [3]:
# Section 1: Build compact user config for zero-shot design workflow.
user_inputs = build_user_inputs(
    root_key="examples",
    data_subfolder="ET096_R0",
    filename_prefix="ET096_",
    wt_sequence="",
    plm_models=["esm2-650m", "esmc-600m", "poet2"],
    marginal_type="masked",
    score_types_to_run={
        "plm_llr": True,
        "proteinmpnn": False,
        "spurs": False,
        "structure_annotations": True,
    },
    pos_to_exclude=[],
    allowed_positions=[],
    allowed_mut_aas=[],
    dist_to_lig_thres=None,
    dist_to_lig_filter_direction="le",
    top_n=100,
    max_num_mut_per_pos=3,
)

# Section 2: Optional filename hooks.
user_inputs["wt_sequence_filename"] = ""
user_inputs["candidate_sequences_filename"] = ""
user_inputs["conservation_filename"] = ""
user_inputs["structure_filename"] = ""
user_inputs["ligand_filename"] = ""
user_inputs["llr_cache_vect_filename_prefix"] = ""

pprint(user_inputs)


{'allowed_mut_aas': [],
 'allowed_positions': [],
 'candidate_sequences_filename': '',
 'conservation_filename': '',
 'data_subfolder': 'ET096_R0',
 'dist_to_lig_filter_direction': 'le',
 'dist_to_lig_thres': None,
 'filename_prefix': 'ET096_',
 'ligand_filename': '',
 'llr_cache_vect_filename_prefix': '',
 'marginal_type': 'masked',
 'max_num_mut_per_pos': 3,
 'plm_models': ['esm2-650m', 'esmc-600m', 'poet2'],
 'pos_to_exclude': [],
 'root_key': 'examples',
 'score_types_to_run': {'plm_llr': True,
                        'plm_meanpll': False,
                        'proteinmpnn': False,
                        'spurs': False,
                        'structure_annotations': True},
 'structure_filename': '',
 'top_n': 100,
 'wt_sequence': '',
 'wt_sequence_filename': ''}


In [4]:
# Section 1: Run the overall workflow driver (prototype).
result = run_zero_shot_mutant_design_workflow(user_inputs, repo_root=repo_root)
print("status:", result.get("status"))

# Section 2: Print target score paths and existence by score group.
import pandas as pd
artifact_df = pd.DataFrame(result.get("artifact_status", []))
if artifact_df.empty:
    print("No score artifacts were requested.")
else:
    def _score_group(name: str) -> str:
        txt = str(name)
        if "LLR vect" in txt:
            return "PLM LLR"
        if "meanPLL" in txt:
            return "PLM meanPLL"
        if "ProteinMPNN" in txt:
            return "ProteinMPNN"
        if "SPURS" in txt:
            return "SPURS"
        if "annotations" in txt:
            return "Structure Annotations"
        return "Other"

    artifact_df["score_group"] = artifact_df["artifact"].map(_score_group)
    for grp, grp_df in artifact_df.groupby("score_group", dropna=False):
        print(f"\n[{grp}]")
        for _, row in grp_df.iterrows():
            status = "FOUND" if bool(row.get("exists", False)) else "MISSING"
            print(f"- {row['artifact']}: {status}")
            print(f"  target: {row['path']}")

# Section 3: Print placeholder step plan.
print("\nsteps_to_run:", len(result.get("steps_to_run", [])))
for s in result.get("steps_to_run", []):
    print(f"- {s.get('step')}: {s.get('reason')} -> {s.get('target_path')}")


status: ok

[Other]
- esm2-650m LLR map png: FOUND
  target: /Users/charmainechia/Documents/projects/agentic-protein-design/examples/encodings/LLR/ET096_esm2-650m_LLR-masked_map.png
- esmc-600m LLR map png: FOUND
  target: /Users/charmainechia/Documents/projects/agentic-protein-design/examples/encodings/LLR/ET096_esmc-600m_LLR-masked_map.png
- poet2 LLR map png: FOUND
  target: /Users/charmainechia/Documents/projects/agentic-protein-design/examples/encodings/LLR/ET096_poet2_LLR-masked_map.png

[PLM LLR]
- esm2-650m LLR vect: FOUND
  target: /Users/charmainechia/Documents/projects/agentic-protein-design/examples/encodings/LLR/ET096_esm2-650m_LLR-masked_vect.csv
- esmc-600m LLR vect: FOUND
  target: /Users/charmainechia/Documents/projects/agentic-protein-design/examples/encodings/LLR/ET096_esmc-600m_LLR-masked_vect.csv
- poet2 LLR vect: FOUND
  target: /Users/charmainechia/Documents/projects/agentic-protein-design/examples/encodings/LLR/ET096_poet2_LLR_vect.csv

[Structure Annotations]
-

In [ ]:
# Section 4: Inspect loaded/compiled scores and shortlist previews.
scores_long_preview = pd.DataFrame(result.get("scores_long_preview", []))
compiled_preview = pd.DataFrame(result.get("compiled_scores_preview", []))
shortlist_preview = pd.DataFrame(result.get("shortlist_preview", []))
scores_long_preview.head(20), compiled_preview.head(20), shortlist_preview.head(20)


## Next Iteration

- Replace placeholder `steps_to_run` entries with real calls to step scripts in `src/agentic_protein_design/steps/`.
- Add structure/SPURS integrations and annotation joins.
- Expand selection to include distance-based and diversity-aware filters.
